## Task 6 Fine-tune a pretrained model 



In [2]:
import os
import sys
import transformers
import pandas as pd
import tensorflow as tf
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import TFAutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import AdamWeightDecay
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM


## loading the pretrained model

In [3]:
model_checkpoint = "Helsinki-NLP/opus-mt-ar-en" #fetching pretrained translation model from hugging face

### loading the dataset from hugging face

In [11]:
from datasets import load_dataset

# Load the dataset
raw_datasets = load_dataset("Amr-khaled/Egyptian-Arabic_English_V1")

# getting arabic dataset


README.md:   0%|          | 0.00/790 [00:00<?, ?B/s]

(…)gyText_Translated-00000-of-00001.parquet:   0%|          | 0.00/828k [00:00<?, ?B/s]

(…)n_Token_EGY_Songs-00000-of-00001.parquet:   0%|          | 0.00/6.80M [00:00<?, ?B/s]

ArzEn_MultiGenre-00000-of-00001.parquet:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

Generating NADI_2024_SubTask_EgyText_Translated split:   0%|          | 0/12799 [00:00<?, ? examples/s]

Generating Milion_Token_EGY_Songs split:   0%|          | 0/6554 [00:00<?, ? examples/s]

Generating ArzEn_MultiGenre split:   0%|          | 0/13946 [00:00<?, ? examples/s]

### checking the subsets of the dataset

In [14]:
print(raw_datasets.keys())


dict_keys(['NADI_2024_SubTask_EgyText_Translated', 'Milion_Token_EGY_Songs', 'ArzEn_MultiGenre'])


### splitting the dataset into trainning and testing and validation

In [15]:
# Load the dataset from the chosen split
train_test_split = raw_datasets['NADI_2024_SubTask_EgyText_Translated'].train_test_split(test_size=0.01)  # 1% for test

# Further split the training set to create validation set (10% of 90% => 9% of original)
validation_test_split = train_test_split['train'].train_test_split(test_size=0.01)  # 1% for validation

# Organizing the splits into a new DatasetDict
raw_datasets = DatasetDict({
    'train': validation_test_split['train'],
    'validation': validation_test_split['test'],
    'test': train_test_split['test']
})

# Print the resulting splits to check
print(raw_datasets)


DatasetDict({
    train: Dataset({
        features: ['Egy', 'English', 'Egy_Text_Source'],
        num_rows: 12544
    })
    validation: Dataset({
        features: ['Egy', 'English', 'Egy_Text_Source'],
        num_rows: 127
    })
    test: Dataset({
        features: ['Egy', 'English', 'Egy_Text_Source'],
        num_rows: 128
    })
})


In [16]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [26]:
print(raw_datasets.keys())


dict_keys(['train', 'validation', 'test'])


In [27]:
print(raw_datasets["train"][:2])  # Check the first 2 examples in the 'train' split


{'Egy': ['ايه رأيك تروح معايا ل هاكوني؟', 'ممكن شوكة جديدة؟'], 'English': ['What do you think, want to go to Hakuna with me?', 'Can I have a new fork?'], 'Egy_Text_Source': ['[NADI 2024: The Fifth Nuanced Arabic Dialect Identification Shared Task](https://aclanthology.org/2024.arabicnlp-1.79) (Abdul-Mageed et al., ArabicNLP-WS 2024)', '[NADI 2024: The Fifth Nuanced Arabic Dialect Identification Shared Task](https://aclanthology.org/2024.arabicnlp-1.79) (Abdul-Mageed et al., ArabicNLP-WS 2024)']}


## Preprocessing Summary

- Translates from Arabic (`Egy`) to English (`English`).
- Tokenizes both input (Arabic) and target (English) text.
- Limits each to 128 tokens max.
- Returns tokenized inputs and target labels for model traini

In [37]:
max_input_length = 128
max_target_length = 128 #max 128 tokens

source_lang = "ar"
target_lang = "en" 
def preprocess_function(examples):
    # Accessing the 'Egy' column for Arabic (source language) and 'English' column for English (target language)
    inputs = examples["Egy"]  # Arabic text
    targets = examples["English"]  # English text

    # Tokenizing the inputs (Arabic sentences)
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # Tokenizing the targets (English sentences) with target tokenizer
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs  # Corrected the typo here

## Testing Preprocessing

- Applies `preprocess_function` to the first 2 examples from the **train** split of `raw_datasets`.
- Useful for quickly verifying that preprocessing works as expected.


In [38]:
# Test on the first 2 examples of the 'train' split (adjust as needed)
preprocess_function(raw_datasets["train"][:2])

{'input_ids': [[1899, 36, 1999, 57, 102, 14045, 82, 8705, 70, 23, 54, 3681, 41, 55, 0], [2500, 30658, 40, 1041, 55, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]], 'labels': [[202, 155, 32, 441, 4, 439, 8, 191, 8, 817, 3559, 21360, 37, 107, 34, 0], [1903, 25, 71, 14, 325, 20, 1279, 34, 0]]}

## Tokenizing the Dataset

- Applies `preprocess_function` to the entire dataset using `.map()` with batching.
- Converts Arabic–English translation pairs into tokenized tensors.
- Prepares data for


In [39]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True) #converts es to en translation pairs into tokenized tensors that the model can learn from 

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

Map:   0%|          | 0/127 [00:00<?, ? examples/s]

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

In [40]:
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_checkpoint) # loads the appropriate architecure automatically 

2025-04-09 08:55:30.013167: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1744188930.013735    2140 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 42532 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:41:00.0, compute capability: 8.9
All model checkpoint layers were used when initializing TFMarianMTModel.

All the layers of TFMarianMTModel were initialized from the model checkpoint at Helsinki-NLP/opus-mt-ar-en.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.


## Training Hyperparameters

- `batch_size = 16`: Number of samples per training step.
- `learning_rate = 2e-5`: Controls how much the model updates during training.
- `weight_decay = 0.01`: Regularization to prevent overfitting.
- `num_train_epochs = 2`: Number of full passes over the training dataset.



In [41]:
batch_size = 16
learning_rate = 2e-5
weight_decay = 0.01
num_train_epochs = 2

## Data Collator

- Uses `DataCollatorForSeq2Seq` to:
  - Format and pad batches of tokenized data.
  - Ensure inputs are the correct shape for the model.
- Returns tensors in TensorFlow format (`return_tensors="tf"`).


In [42]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors="tf") #formats and pads a list of examples (the data)

## Generation Data Collator

- Similar to the regular `DataCollatorForSeq2Seq`.
- Pads sequences so their lengths are multiples of 128 (`pad_to_multiple_of=128`).
- Useful for efficient generation and better performance on some hardware (like TPUs/GPUs).
- Outputs TensorFlow tensors.


In [43]:
generation_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors="tf", pad_to_multiple_of=128)

## Dataset Preparation

- **Training Dataset**:
  - Uses the `"test"` split temporarily (should be `"train"`).
  - Prepares batches with `data_collator`.
  - Shuffled for training.

- **Validation Dataset**:
  - Uses the `"validation"` split.
  - Not shuffled.
  - Used during training to monitor performance.

- **Generation Dataset**:
  - Also uses `"validation"` split.
  - Uses `generation_data_collator` with `pad_to_multiple_of=128`.
  - Intended for inference (e.g., generating translations).


In [44]:
train_dataset = model.prepare_tf_dataset( #creates and prepares the dataset for training
    tokenized_datasets["test"], #it uses the test because the dataset is so large it would take far too long to train but it should be "train"
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator,
)

validation_dataset = model.prepare_tf_dataset( #this one is user for validatioin in training
    tokenized_datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    collate_fn=data_collator,
)

generation_dataset = model.prepare_tf_dataset( #uses validation set but for inference
    tokenized_datasets["validation"],
    batch_size=8,
    shuffle=False,
    collate_fn=generation_data_collator,
)

## Optimizer Setup

- Uses `AdamWeightDecay`:
  - A variant of Adam optimizer with weight decay regularization.
  - Helps prevent overfitting.
- Compiles the model with this optimizer for training.


In [45]:
optimizer = AdamWeightDecay(learning_rate=learning_rate, weight_decay_rate=weight_decay)
model.compile(optimizer=optimizer)

## Model Training

- Trains the model using `train_dataset`.
- Validates on `validation_dataset` after each epoch.
- Runs for 3 epochs.


In [47]:
model.fit(train_dataset, validation_data=validation_dataset, epochs=3) #training

Epoch 1/3
8/8 [==============================] - 1s 123ms/step - loss: 1.1930 - val_loss: 1.1676
Epoch 2/3
8/8 [==============================] - 1s 124ms/step - loss: 0.9350 - val_loss: 1.1289
Epoch 3/3
8/8 [==============================] - 1s 130ms/step - loss: 0.7712 - val_loss: 1.1095


## Saving Model and Tokenizer

- Saves the model's configuration and weights to `./arabic_en`.
- Saves the tokenizer files to `./arabic_en` for future use or inference.


In [51]:
model.save_pretrained("./arabic_en")         # saves config + weights
tokenizer.save_pretrained("./arabic_en")     # saves tokenizer files


/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:397: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[62833]]}
  warnings.warn(


('./arabic_en/tokenizer_config.json',
 './arabic_en/special_tokens_map.json',
 './arabic_en/vocab.json',
 './arabic_en/source.spm',
 './arabic_en/target.spm',
 './arabic_en/added_tokens.json')

## Loading Model and Tokenizer

- Loads the pre-trained model from the `./arabic_en` directory using `TFAutoModelForSeq2SeqLM`.
- Loads the tokenizer from the same directory using `AutoTokenizer`.



In [52]:
model = TFAutoModelForSeq2SeqLM.from_pretrained("./arabic_en")
tokenizer = AutoTokenizer.from_pretrained("./arabic_en") #loading model and tokenizer

All model checkpoint layers were used when initializing TFMarianMTModel.

All the layers of TFMarianMTModel were initialized from the model checkpoint at ./arabic_en.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.
/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


###  ## Translation Pipeline

1. **Loading Model and Tokenizer**:
   - Loads the pre-trained model and tokenizer from the `./arabic_en` directory.

2. **Translation Function**:
   - Defines a `translate` function that:
     - Tokenizes the input text.
     - Generates a translation using the model.
     - Decodes the translated output.

3. **Applying Translation**:
   - Reads a CSV file (`whisper1_processed.csv`).
   - Applies the `translate` function to the `Sentence` column and stores translations in a new column (`translated_sentence`).

4. **Saving Output**:
   - Saves the translated sentences to a new CSV file (`whisper1_translated.csv`).


In [53]:
model_path = "./arabic_en"  
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_path)

input_path = "../task6/whisper1_processed.csv"    # <-- Replace with your Excel file path
df = pd.read_csv(input_path)

def translate(text):

    # Tokenize input text
    inputs = tokenizer.encode(text, return_tensors="tf", padding=True, truncation=True, max_length=256)
    # Generate translation
    outputs = model.generate(inputs, max_length=256)
    # Decode the output
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Apply translation to the column 
df['translated_sentence'] = df['Sentence'].apply(translate)

# save to excel file
output_path = "../task6/whisper1_translatedd.csv"   # <-- Replace with desired output path
df.to_csv(output_path, index=False)


All model checkpoint layers were used when initializing TFMarianMTModel.

All the layers of TFMarianMTModel were initialized from the model checkpoint at ./arabic_en.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.
